<a href="https://www.kaggle.com/code/rudrapratapsingh1534/spectra-llm?scriptVersionId=314620704" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import os
import math
import glob
import time
import shutil
import torch
import torch.nn as nn
import torch.nn.functional as F

from dataclasses import dataclass
from contextlib import contextmanager

from torch.utils.data import Dataset, DataLoader, Subset

from transformers import GPT2TokenizerFast
from datasets import load_dataset
from torch.amp import autocast, GradScaler

MODE = "train"
DEBUG = True

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

device = "cuda" if torch.cuda.is_available() else "cpu"

print("device:", device)
if device == "cuda":
    print(torch.cuda.get_device_name(0))

os.makedirs("/kaggle/output", exist_ok=True)

device: cuda
Tesla T4


In [2]:
@dataclass
class Config:

    # ---------- DATA ----------
    tokens_dir: str = "/kaggle/working/tokens"
    ckpt_path: str = "/kaggle/working/ckpt_latest.pt"
    best_ckpt_path: str = "/kaggle/working/ckpt_best.pt"

    # ---------- MODEL ----------
    vocab_size: int = 50257
    block_size: int = 768

    n_embd: int = 768
    n_head: int = 12
    n_layer: int = 12
    dropout: float = 0.0

    # ---------- TRAIN ----------
    batch_size: int = 20
    grad_accum_steps: int = 4

    max_steps: int = 12001
    warmup_steps: int = 1000

    max_lr: float = 1e-4
    min_lr: float = 1e-5

    weight_decay: float = 0.1
    grad_clip: float = 1.0

    # ---------- LOGGING ----------
    log_every: int = 50
    sample_every: int = 2000
    save_every: int = 4000

    # ---------- VALIDATION ----------
    val_fraction: float = 0.01
    eval_every: int = 1000
    eval_batches: int = 50

cfg = Config()
cfg

Config(tokens_dir='/kaggle/working/tokens', ckpt_path='/kaggle/working/ckpt_latest.pt', best_ckpt_path='/kaggle/working/ckpt_best.pt', vocab_size=50257, block_size=768, n_embd=768, n_head=12, n_layer=12, dropout=0.0, batch_size=20, grad_accum_steps=4, max_steps=12001, warmup_steps=1000, max_lr=0.0001, min_lr=1e-05, weight_decay=0.1, grad_clip=1.0, log_every=50, sample_every=2000, save_every=4000, val_fraction=0.01, eval_every=1000, eval_batches=50)

In [3]:
def build_tokens(
    cfg,
    chunk_size=7_500_000,
    target_tokens=750_000_000
):
    os.makedirs(cfg.tokens_dir, exist_ok=True)

    existing = sorted(glob.glob(f"{cfg.tokens_dir}/tokens_*.pt"))

    if len(existing) > 0:
        print("token shards already exist:")
        print(existing)
        return

    tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

    dataset = load_dataset(
        "HuggingFaceFW/fineweb-edu-score-2",
        split="train",
        streaming=True
    )

    print("streaming FineWeb-Edu-Score-2...")

    buffer = []
    shard_idx = 0
    total_tokens = 0

    for i, row in enumerate(dataset):

        text = row["text"]

        if DEBUG and i == 0:
            print("\nRAW SAMPLE:\n")
            print(text[:1000])

        ids = tokenizer(
            text,
            add_special_tokens=True,
            truncation=False
        )["input_ids"]

        buffer.extend(ids)
        total_tokens += len(ids)

        if len(buffer) >= chunk_size:

            path = f"{cfg.tokens_dir}/tokens_{shard_idx}.pt"

            torch.save(
                torch.tensor(buffer, dtype=torch.long),
                path
            )

            print(f"saved {path} | tokens={len(buffer):,}")

            buffer = []
            shard_idx += 1

        if i % 1000 == 0:
            print(f"docs={i:,} | tokens={total_tokens:,}")

        if total_tokens >= target_tokens:
            break

    if len(buffer) > 0:
        path = f"{cfg.tokens_dir}/tokens_{shard_idx}.pt"

        torch.save(
            torch.tensor(buffer, dtype=torch.long),
            path
        )

        print(f"saved final {path}")

    print(f"done | total tokens={total_tokens:,}")

In [4]:
class ShardedDataset(Dataset):

    def __init__(
        self,
        token_files,
        block_size,
        window_size=50
    ):
        assert len(token_files) > 0

        self.token_files = token_files
        self.block_size = block_size
        self.window_size = min(window_size, len(token_files))

        self.window_start = 0

        self.load_window()

    def load_window(self):

        start = self.window_start
        end = start + self.window_size

        self.active_files = self.token_files[start:end]

        print(f"\nLoading shards {start} -> {end-1}")

        self.shards = [
            torch.load(f, weights_only=True)
            for f in self.active_files
        ]

        self.lengths = [len(s) for s in self.shards]

        self.cum = []
        total = 0

        for l in self.lengths:
            total += l
            self.cum.append(total)

        self.total_tokens = total

        print(
            f"Loaded {len(self.shards)} shards | "
            f"{self.total_tokens:,} tokens"
        )

    def slide_window(self):

        if self.window_start + self.window_size >= len(self.token_files):
            return False

        self.window_start += 1
        self.load_window()
        return True

    def __len__(self):
        return self.total_tokens - self.block_size - 1

    def _find_shard(self, idx):

        lo, hi = 0, len(self.cum) - 1

        while lo < hi:
            mid = (lo + hi) // 2

            if idx < self.cum[mid]:
                hi = mid
            else:
                lo = mid + 1

        return lo

    def __getitem__(self, idx):

        shard_idx = self._find_shard(idx)

        shard = self.shards[shard_idx]

        local_idx = idx - (
            self.cum[shard_idx - 1]
            if shard_idx > 0 else 0
        )

        if local_idx + self.block_size + 1 <= len(shard):

            x = shard[
                local_idx:
                local_idx + self.block_size
            ]

            y = shard[
                local_idx + 1:
                local_idx + self.block_size + 1
            ]

            return x, y

        return self.__getitem__((idx + 1) % len(self))

In [5]:
class CausalSelfAttention(nn.Module):

    def __init__(self, cfg):
        super().__init__()

        assert cfg.n_embd % cfg.n_head == 0

        self.n_head = cfg.n_head
        self.head_dim = cfg.n_embd // cfg.n_head
        self.dropout = cfg.dropout

        self.qkv = nn.Linear(
            cfg.n_embd,
            3 * cfg.n_embd,
            bias=False
        )

        self.proj = nn.Linear(
            cfg.n_embd,
            cfg.n_embd,
            bias=False
        )

        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):

        B, T, C = x.shape

        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=-1)

        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        out = F.scaled_dot_product_attention(
            q, k, v,
            is_causal=True,
            dropout_p=self.dropout if self.training else 0.0
        )

        out = out.transpose(1, 2).contiguous().view(B, T, C)

        out = self.proj(out)
        out = self.drop(out)

        return out


class MLP(nn.Module):

    def __init__(self, cfg):
        super().__init__()

        self.fc1 = nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=False)
        self.fc2 = nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=False)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):

        x = self.fc1(x)
        x = F.gelu(x, approximate="tanh")
        x = self.fc2(x)
        x = self.drop(x)

        return x


class Block(nn.Module):

    def __init__(self, cfg):
        super().__init__()

        self.ln1 = nn.LayerNorm(cfg.n_embd)
        self.ln2 = nn.LayerNorm(cfg.n_embd)

        self.attn = CausalSelfAttention(cfg)
        self.mlp = MLP(cfg)

    def forward(self, x):

        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))

        return x


class SpectraLLM(nn.Module):

    def __init__(self, cfg):
        super().__init__()

        self.config = cfg

        self.transformer = nn.ModuleDict({

            "tok_emb": nn.Embedding(
                cfg.vocab_size,
                cfg.n_embd
            ),

            "pos_emb": nn.Embedding(
                cfg.block_size,
                cfg.n_embd
            ),

            "drop": nn.Dropout(cfg.dropout),

            "blocks": nn.ModuleList(
                [Block(cfg) for _ in range(cfg.n_layer)]
            ),

            "ln_f": nn.LayerNorm(cfg.n_embd)
        })

        self.lm_head = nn.Linear(
            cfg.n_embd,
            cfg.vocab_size,
            bias=False
        )

        self.lm_head.weight = self.transformer["tok_emb"].weight

        self.apply(self._init_weights)

        for name, p in self.named_parameters():
            if name.endswith("proj.weight") or name.endswith("fc2.weight"):
                nn.init.normal_(
                    p,
                    std=0.02 / math.sqrt(2 * cfg.n_layer)
                )

        n_params = sum(p.numel() for p in self.parameters())
        print(f"params: {n_params/1e6:.1f}M")

    def forward(self, idx, targets=None):

        B, T = idx.shape

        pos = torch.arange(T, device=idx.device)

        tok = self.transformer["tok_emb"](idx)
        pos = self.transformer["pos_emb"](pos)

        x = self.transformer["drop"](tok + pos)

        for block in self.transformer["blocks"]:
            x = block(x)

        x = self.transformer["ln_f"](x)

        logits = self.lm_head(x)

        loss = None

        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )

        return logits, loss

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=0.02)

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=0.02)

In [6]:
def build_optimizer(model, cfg):

    decay = []
    no_decay = []

    for name, p in model.named_parameters():

        if not p.requires_grad:
            continue

        if (
            p.ndim < 2
            or "ln" in name
            or "emb" in name
            or "bias" in name
        ):
            no_decay.append(p)
        else:
            decay.append(p)

    try:
        optimizer = torch.optim.AdamW(
            [
                {"params": decay, "weight_decay": cfg.weight_decay},
                {"params": no_decay, "weight_decay": 0.0},
            ],
            lr=cfg.max_lr,
            betas=(0.9, 0.95),
            fused=True
        )

    except:
        optimizer = torch.optim.AdamW(
            [
                {"params": decay, "weight_decay": cfg.weight_decay},
                {"params": no_decay, "weight_decay": 0.0},
            ],
            lr=cfg.max_lr,
            betas=(0.9, 0.95)
        )

    return optimizer


def get_lr(step, cfg):

    if step < cfg.warmup_steps:
        return cfg.max_lr * step / cfg.warmup_steps

    if step >= cfg.max_steps:
        return cfg.min_lr

    progress = (
        step - cfg.warmup_steps
    ) / (
        cfg.max_steps - cfg.warmup_steps
    )

    cosine = 0.5 * (
        1 + math.cos(math.pi * progress)
    )

    return cfg.min_lr + cosine * (
        cfg.max_lr - cfg.min_lr
    )

In [7]:
def save_checkpoint(
    model,
    optimizer,
    scaler,
    step,
    path
):
    tmp_path = path + ".tmp"

    ckpt = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "step": step
    }

    # write temp first
    torch.save(ckpt, tmp_path)

    # remove old file if exists
    if os.path.exists(path):
        os.remove(path)

    # atomic rename
    os.rename(tmp_path, path)

    print(f"saved checkpoint @ step {step} -> {path}")

def load_checkpoint(
    model,
    optimizer,
    scaler,
    path,
    device
):

    ckpt = torch.load(
        path,
        map_location=device
    )

    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scaler.load_state_dict(ckpt["scaler"])

    step = ckpt["step"]

    print(f"resumed from step {step}")

    return step

In [8]:
@torch.no_grad()
def generate(
    model,
    tokenizer,
    prompt="Once upon a time",
    max_new_tokens=80,
    temperature=0.8,
    top_p=0.9
):

    model.eval()

    tokens = tokenizer.encode(
        prompt,
        return_tensors="pt"
    ).to(device)

    for _ in range(max_new_tokens):

        x = tokens[:, -model.config.block_size:]

        with autocast("cuda", dtype=torch.float16):
            logits, _ = model(x)

        logits = logits[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)

        sorted_probs, sorted_idx = torch.sort(
            probs,
            descending=True
        )

        cumulative = torch.cumsum(
            sorted_probs,
            dim=-1
        )

        mask = cumulative > top_p
        mask[..., 1:] = mask[..., :-1].clone()
        mask[..., 0] = False

        sorted_probs[mask] = 0
        sorted_probs /= sorted_probs.sum(
            dim=-1,
            keepdim=True
        )

        next_token = sorted_idx.gather(
            -1,
            torch.multinomial(sorted_probs, 1)
        )

        tokens = torch.cat(
            [tokens, next_token],
            dim=1
        )

    text = tokenizer.decode(
        tokens[0].tolist(),
        skip_special_tokens=True
    )

    model.train()

    return text


@torch.no_grad()
def evaluate(
    model,
    loader,
    cfg
):

    model.eval()

    losses = []

    for i, (x, y) in enumerate(loader):

        if i >= cfg.eval_batches:
            break

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with autocast("cuda", dtype=torch.float16):
            _, loss = model(x, y)

        losses.append(loss.item())

    model.train()

    return sum(losses) / len(losses)

In [9]:
def train():

    token_files = sorted(
        glob.glob(f"{cfg.tokens_dir}/tokens_*.pt")
    )

    if len(token_files) == 0:
        build_tokens(cfg)

        token_files = sorted(
            glob.glob(f"{cfg.tokens_dir}/tokens_*.pt")
        )

    dataset = ShardedDataset(
        token_files,
        cfg.block_size,
        window_size=50
    )

    n_total = len(dataset)
    n_val = int(n_total * cfg.val_fraction)
    n_train = n_total - n_val

    train_ds = Subset(
        dataset,
        range(0, n_train)
    )

    val_ds = Subset(
        dataset,
        range(n_train, n_total)
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=3,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=4
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=1,
        pin_memory=True
    )

    tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

    model = SpectraLLM(cfg).to(device)

    model = torch.compile(model)

    optimizer = build_optimizer(model, cfg)

    scaler = GradScaler("cuda")

    start_step = 0
    best_val = float("inf")

    if os.path.exists(cfg.ckpt_path):

        start_step = load_checkpoint(
            model,
            optimizer,
            scaler,
            cfg.ckpt_path,
            device
        )

    model.train()

    data_iter = iter(train_loader)

    t0 = time.perf_counter()

    for step in range(start_step, cfg.max_steps):

        lr = get_lr(step, cfg)

        for pg in optimizer.param_groups:
            pg["lr"] = lr

        optimizer.zero_grad(set_to_none=True)

        loss_accum = 0.0

        for _ in range(cfg.grad_accum_steps):

            try:
                x, y = next(data_iter)

            except StopIteration:

                moved = dataset.slide_window()

                if moved:
                    print("\nSliding shard window...\n")

                else:
                    print("\nFinal shard window reached. Restarting.\n")

                n_total = len(dataset)
                n_val = int(n_total * cfg.val_fraction)
                n_train = n_total - n_val

                train_ds = Subset(
                    dataset,
                    range(0, n_train)
                )

                val_ds = Subset(
                    dataset,
                    range(n_train, n_total)
                )

                train_loader = DataLoader(
                    train_ds,
                    batch_size=cfg.batch_size,
                    shuffle=True,
                    num_workers=3,
                    pin_memory=True,
                    persistent_workers=True,
                    prefetch_factor=4
                )

                val_loader = DataLoader(
                    val_ds,
                    batch_size=cfg.batch_size,
                    shuffle=False,
                    num_workers=1,
                    pin_memory=True
                )

                data_iter = iter(train_loader)

                x, y = next(data_iter)

            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            with autocast("cuda", dtype=torch.float16):

                _, loss = model(x, y)

                loss = loss / cfg.grad_accum_steps

            scaler.scale(loss).backward()

            loss_accum += loss.item()

        scaler.unscale_(optimizer)

        grad_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            cfg.grad_clip
        )

        scaler.step(optimizer)

        scaler.update()

        if step % cfg.log_every == 0:

            dt = time.perf_counter() - t0

            print(
                f"step {step:5d} | "
                f"loss {loss_accum:.4f} | "
                f"grad {grad_norm:.2f} | "
                f"lr {lr:.2e} | "
                f"{dt:.2f}s"
            )

            t0 = time.perf_counter()

        if step % cfg.eval_every == 0 and step > 0:

            val_loss = evaluate(
                model,
                val_loader,
                cfg
            )

            print(
                f"VAL step {step:5d} | "
                f"loss {val_loss:.4f}"
            )

            if step % 1000 == 0 and val_loss < best_val - 0.02:

                best_val = val_loss

                save_checkpoint(
                    model,
                    optimizer,
                    scaler,
                    step,
                    cfg.best_ckpt_path
                )

                shutil.copy2(
                    cfg.best_ckpt_path,
                    "/kaggle/output/ckpt_best.pt"
                )

                print(f"NEW BEST @ step {step}")

        if step % cfg.sample_every == 0 and step > 0:

            print("\n================================")

            print(generate(model, tokenizer))

            print("================================\n")

        if step % cfg.save_every == 0 and step > 0:

            save_checkpoint(
                model,
                optimizer,
                scaler,
                step,
                cfg.ckpt_path
            )

    return model

In [ ]:
if MODE == "build_tokens":
    build_tokens(cfg)

elif MODE == "train":
    try:
        model = train()

    except Exception as e:
        print("=== FATAL ERROR ===")
        traceback.print_exc()

        # optional backup latest checkpoint
        try:
            shutil.copy2(
                cfg.ckpt_path,
                "/kaggle/output/ckpt_crash.pt"
            )
            print("Crash checkpoint copied.")
        except:
            pass

        sys.exit(1)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

streaming FineWeb-Edu-Score-2...


Token indices sequence length is longer than the specified maximum sequence length for this model (1240 > 1024). Running this sequence through the model will result in indexing errors



RAW SAMPLE:

How AP reported in all formats from tornado-stricken regionsMarch 8, 2012
When the first serious bout of tornadoes of 2012 blew through middle America in the middle of the night, they touched down in places hours from any AP bureau. Our closest video journalist was Chicago-based Robert Ray, who dropped his plans to travel to Georgia for Super Tuesday, booked several flights to the cities closest to the strikes and headed for the airport. He’d decide once there which flight to take.
He never got on board a plane. Instead, he ended up driving toward Harrisburg, Ill., where initial reports suggested a town was destroyed. That decision turned out to be a lucky break for the AP. Twice.
Ray was among the first journalists to arrive and he confirmed those reports -- in all formats. He shot powerful video, put victims on the phone with AP Radio and played back sound to an editor who transcribed the interviews and put the material on text wires. He then walked around the devastati

W0426 17:33:47.175000 55 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


step     0 | loss 10.9979 | grad 15.17 | lr 0.00e+00 | 54.86s
step    50 | loss 9.5809 | grad 2.24 | lr 5.00e-06 | 161.59s
step   100 | loss 9.0133 | grad 1.97 | lr 1.00e-05 | 175.44s
step   150 | loss 8.5219 | grad 1.59 | lr 1.50e-05 | 175.80s
step   200 | loss 8.0180 | grad 1.55 | lr 2.00e-05 | 175.63s
step   250 | loss 7.3962 | grad 1.12 | lr 2.50e-05 | 175.18s
step   300 | loss 7.0782 | grad 0.96 | lr 3.00e-05 | 175.25s
step   350 | loss 6.8958 | grad 1.00 | lr 3.50e-05 | 175.73s
step   400 | loss 6.7860 | grad 1.42 | lr 4.00e-05 | 175.22s
step   450 | loss 6.7050 | grad 1.55 | lr 4.50e-05 | 175.98s
step   500 | loss 6.4980 | grad 1.72 | lr 5.00e-05 | 175.94s
step   550 | loss 6.4724 | grad 1.02 | lr 5.50e-05 | 175.62s
step   600 | loss 6.3973 | grad 1.37 | lr 6.00e-05 | 175.51s
step   650 | loss 6.2786 | grad 1.47 | lr 6.50e-05 | 175.74s
step   700 | loss 6.2150 | grad 2.10 | lr 7.00e-05 | 175.85s
step   750 | loss 6.1930 | grad 1.23 | lr 7.50e-05 | 175.71s
step   800 | loss 6.144